# Módulo 08: NumPy

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m08-numpy.ipynb](m08-numpy.ipynb). No es una tarea entregada del programa.

Las listas de Python sirven para casi todo, pero son lentas cuando hay que hacer la misma operación sobre miles o millones de números. **NumPy** resuelve ese problema con el arreglo `ndarray`: una tabla de valores del mismo tipo guardados de forma contigua en memoria, sobre la que las operaciones se ejecutan en código compilado. Casi toda la pila de ciencia de datos (pandas, matplotlib, scikit-learn) se apoya en él.

1. Por qué NumPy
2. Crear arreglos: `array`, `arange`, `linspace`
3. Indexado y slicing
4. Operaciones vectorizadas y broadcasting
5. Indexado booleano
6. Combinar y comparar arreglos
7. Arreglos 2D, `reshape` y `axis`
8. Errores frecuentes
9. Ejercicio

## 1. Por qué NumPy

Con una lista, sumar 1 a cada elemento exige un bucle: Python interpreta cada iteración, comprueba el tipo de cada número y crea un objeto nuevo. Con un arreglo, la expresión `x + 1` se aplica a todos los elementos a la vez en código C. A esto se le llama **vectorización**: escribir la operación sobre el arreglo completo en lugar de recorrerlo.

El módulo `timeit` mide cuánto tarda un fragmento de código. Los tiempos exactos dependen del equipo, pero la diferencia suele ser de un orden de magnitud o más.

In [1]:
import timeit

import numpy as np

numeros = list(range(100_000))
arreglo = np.arange(100_000)

# Cada fragmento se ejecuta 20 veces; timeit devuelve el tiempo total en segundos
t_lista = timeit.timeit(lambda: [n * 2 for n in numeros], number=20)
t_numpy = timeit.timeit(lambda: arreglo * 2, number=20)

print(f"Lista (bucle):  {t_lista:.4f} s")
print(f"NumPy:          {t_numpy:.4f} s")
print(f"NumPy es unas {t_lista / t_numpy:.0f} veces más rápido en este equipo")

Lista (bucle):  0.0577 s
NumPy:          0.0007 s
NumPy es unas 80 veces más rápido en este equipo


## 2. Crear arreglos: `array`, `arange`, `linspace`

| Función | Resultado |
| --- | --- |
| `np.array(lista)` | Arreglo con los valores de la lista |
| `np.arange(inicio, fin, paso)` | Valores desde `inicio` hasta `fin` (sin incluirlo) con el paso dado |
| `np.linspace(inicio, fin, n)` | `n` valores equiespaciados, **incluyendo** ambos extremos |
| `np.zeros(n)`, `np.ones(n)` | Arreglos de ceros o de unos |

Todo arreglo tiene un **`dtype`** (el tipo de sus elementos, igual para todos) y un **`shape`** (una tupla con el tamaño de cada dimensión).

In [2]:
enteros = np.array([3, 1, 4, 1, 5])
mixto = np.array([3, 1.5, 4])          # un solo float convierte todo a float
pasos = np.arange(0, 10, 2)
malla = np.linspace(0, 1, 5)

for nombre, a in [("enteros", enteros), ("mixto", mixto), ("pasos", pasos), ("malla", malla)]:
    print(f"{nombre:8} {a}  dtype={a.dtype}  shape={a.shape}")

enteros  [3 1 4 1 5]  dtype=int64  shape=(5,)
mixto    [3.  1.5 4. ]  dtype=float64  shape=(3,)
pasos    [0 2 4 6 8]  dtype=int64  shape=(5,)
malla    [0.   0.25 0.5  0.75 1.  ]  dtype=float64  shape=(5,)


Como todos los elementos comparten tipo, NumPy convierte los valores al tipo más general que los represente: en `mixto`, el entero 3 pasó a `3.`. Con `astype` se cambia el tipo de forma explícita.

In [3]:
print(mixto.astype(int))   # astype trunca la parte decimal, no redondea
print(np.round(mixto))     # np.round redondea; los .5 van al par más cercano

[3 1 4]
[3. 2. 4.]


## 3. Indexado y slicing

Los arreglos 1D se indexan igual que las listas: el índice empieza en 0, los índices negativos cuentan desde el final y `a[inicio:fin:paso]` extrae un tramo. Un **paso negativo** recorre el arreglo hacia atrás; `a[::-1]` lo invierte completo.

In [4]:
dias = np.array([10, 20, 30, 40, 50, 60, 70])

print(dias[0], dias[-1])   # primero y último
print(dias[2:5])           # posiciones 2, 3 y 4
print(dias[::2])           # uno sí, uno no
print(dias[::-1])          # invertido
print(dias[5:1:-1])        # de la posición 5 a la 2, hacia atrás

10 70
[30 40 50]
[10 30 50 70]
[70 60 50 40 30 20 10]
[60 50 40 30]


Hay una diferencia importante con las listas: el slicing de un arreglo devuelve una **vista**, no una copia. Modificar la vista modifica el arreglo original. Si se necesita un arreglo independiente, se usa `.copy()`.

In [5]:
tramo = dias[:3]
tramo[0] = 999
print("Original después de modificar la vista:", dias)

dias[0] = 10
independiente = dias[:3].copy()
independiente[0] = -1
print("Original intacto con copy():", dias)

Original después de modificar la vista: [999  20  30  40  50  60  70]
Original intacto con copy(): [10 20 30 40 50 60 70]


## 4. Operaciones vectorizadas y broadcasting

Los operadores aritméticos (`+`, `-`, `*`, `/`, `**`) y las funciones de NumPy (`np.sqrt`, `np.log`, `np.exp`…) actúan elemento a elemento. Así, una fórmula escrita para un número sirve sin cambios para un arreglo. Por ejemplo, convertir kilómetros a millas:

$$\text{millas} = \frac{\text{km}}{1.609}$$

In [6]:
def km_a_millas(km):
    return km / 1.609

distancias_km = np.array([5, 10, 21.1, 42.2])
print(np.round(km_a_millas(distancias_km), 2))
print(round(km_a_millas(100), 2))    # la misma función acepta un número suelto

[ 3.11  6.22 13.11 26.23]
62.15


El **broadcasting** es la regla que permite operar arreglos de tamaños distintos: un escalar se "estira" para encajar con el arreglo, y un arreglo 1D se aplica a cada fila de uno 2D si sus tamaños coinciden con el número de columnas. Si las formas no son compatibles, NumPy lanza un `ValueError`.

In [7]:
precios = np.array([[10.0, 20.0, 30.0],
                    [12.0, 22.0, 33.0]])     # 2 tiendas x 3 productos
descuento = np.array([0.10, 0.0, 0.25])       # uno por producto (columna)

print(precios * 1.16)              # escalar: IVA a todo
print(precios * (1 - descuento))   # 1D: un descuento por columna

[[11.6  23.2  34.8 ]
 [13.92 25.52 38.28]]
[[ 9.   20.   22.5 ]
 [10.8  22.   24.75]]


## 5. Indexado booleano

Una comparación sobre un arreglo devuelve un arreglo de `True`/`False` del mismo tamaño: una **máscara**. Usada como índice, la máscara selecciona los elementos donde vale `True`. Las condiciones se combinan con `&` (y), `|` (o) y `~` (no), siempre entre paréntesis.

In [8]:
lluvia_mm = np.array([0.0, 12.5, 3.2, 0.0, 25.1, 7.8, 0.4])

mascara = lluvia_mm > 5
print(mascara)
print("Días con más de 5 mm:", lluvia_mm[mascara])
print("Entre 1 y 10 mm:", lluvia_mm[(lluvia_mm >= 1) & (lluvia_mm <= 10)])
print("Días sin lluvia:", lluvia_mm[~(lluvia_mm > 0)])

# Como True vale 1, sumar una máscara cuenta cuántos elementos cumplen
print("Días con lluvia:", (lluvia_mm > 0).sum())

[False  True False False  True  True False]
Días con más de 5 mm: [12.5 25.1  7.8]
Entre 1 y 10 mm: [3.2 7.8]
Días sin lluvia: [0. 0.]
Días con lluvia: 5


## 6. Combinar y comparar arreglos

| Función | Resultado |
| --- | --- |
| `np.concatenate((a, b))` | Une los arreglos uno detrás de otro, con repetidos |
| `np.unique(a)` | Valores distintos, **ya ordenados** |
| `np.sort(a)` | Copia ordenada de menor a mayor |
| `np.union1d(a, b)` | Valores que están en `a` o en `b`, sin repetir y ordenados |
| `np.intersect1d(a, b)` | Valores que están en ambos |
| `np.setdiff1d(a, b)` | Valores de `a` que no están en `b` |

In [9]:
lunes = np.array([105, 102, 110, 102])     # identificadores de clientes
martes = np.array([110, 120, 101, 105])

print(np.concatenate((lunes, martes)))
print(np.unique(lunes))
print("Algún día:", np.union1d(lunes, martes))
print("Ambos días:", np.intersect1d(lunes, martes))
print("Solo el lunes:", np.setdiff1d(lunes, martes))

[105 102 110 102 110 120 101 105]
[102 105 110]
Algún día: [101 102 105 110 120]
Ambos días: [105 110]
Solo el lunes: [102]


## 7. Arreglos 2D, `reshape` y `axis`

Un arreglo 2D es una tabla de filas y columnas. Se indexa con `a[fila, columna]`, y cada posición admite slicing: `a[:, 0]` es la primera columna completa. `reshape` reorganiza los mismos datos con otra forma, siempre que el número total de elementos coincida.

In [10]:
ventas = np.arange(1, 13).reshape(3, 4)    # 3 sucursales x 4 trimestres
print(ventas)
print("shape:", ventas.shape)
print("Sucursal 2:", ventas[1])
print("Trimestre 1:", ventas[:, 0])
print("Elemento [2, 3]:", ventas[2, 3])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
shape: (3, 4)
Sucursal 2: [5 6 7 8]
Trimestre 1: [1 5 9]
Elemento [2, 3]: 12


Las agregaciones (`sum`, `mean`, `max`, `min`…) aceptan el argumento **`axis`**, que indica qué dimensión se "colapsa":

| Llamada | Qué hace | Resultado |
| --- | --- | --- |
| `a.sum()` | Suma todo | Un número |
| `a.sum(axis=0)` | Suma a lo largo de las filas | Un valor por **columna** |
| `a.sum(axis=1)` | Suma a lo largo de las columnas | Un valor por **fila** |

In [11]:
print("Total:", ventas.sum())
print("Por trimestre (axis=0):", ventas.sum(axis=0))
print("Por sucursal (axis=1):", ventas.sum(axis=1))
print("Media por sucursal:", ventas.mean(axis=1))

Total: 78
Por trimestre (axis=0): [15 18 21 24]
Por sucursal (axis=1): [10 26 42]
Media por sucursal: [ 2.5  6.5 10.5]


## 8. Errores frecuentes

**Esperar que `+` concatene.** Con listas, `+` las une; con arreglos, suma elemento a elemento. Para unir arreglos se usa `np.concatenate`.

In [12]:
print([1, 2] + [3, 4])
print(np.array([1, 2]) + np.array([3, 4]))
print(np.concatenate((np.array([1, 2]), np.array([3, 4]))))

[1, 2, 3, 4]
[4 6]
[1 2 3 4]


**Usar `and`/`or` con máscaras.** `and` y `or` esperan un único booleano y fallan con un arreglo. Hay que usar `&` y `|`, con cada condición entre paréntesis porque `&` tiene más prioridad que `>`.

In [13]:
edades = np.array([15, 22, 37, 64, 70])
try:
    edades[edades > 18 and edades < 65]
except ValueError as e:
    print("ValueError:", e)
print(edades[(edades > 18) & (edades < 65)])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
[22 37 64]


**Ordenar dos veces.** `np.unique` ya devuelve los valores ordenados, y `np.union1d` ya elimina repetidos y ordena. Envolverlos en `np.sort` no cambia el resultado; solo añade trabajo.

In [14]:
a = np.array([7, 3, 7, 1])
b = np.array([3, 9])
print(np.sort(np.unique(np.concatenate((a, b)))))
print(np.union1d(a, b))    # mismo resultado en una sola llamada

[1 3 7 9]
[1 3 7 9]


**Perder decimales por el `dtype`.** Asignar un float a un arreglo de enteros lo trunca sin avisar. Si se van a guardar decimales, el arreglo debe crearse como float.

In [15]:
conteos = np.array([1, 2, 3])
conteos[0] = 2.9
print(conteos)                             # el 2.9 se guardó como 2
print(np.array([1, 2, 3], dtype=float))

[2 2 3]
[1. 2. 3.]


**Confundir `axis=0` con "por fila".** `axis=0` recorre las filas para producir un resultado **por columna**. Ante la duda, conviene comparar el `shape` del resultado con el del arreglo.

## 9. Ejercicio

Una estación meteorológica registró la temperatura máxima, en °C, de dos semanas:

```python
maximas = np.array([31.2, 29.8, 33.5, 35.1, 34.0, 28.7, 27.9,
                    30.4, 32.6, 36.2, 35.8, 29.1, 28.3, 30.0])
```

1. Convierte las temperaturas a grados Fahrenheit con $F = C \cdot \frac{9}{5} + 32$, sin bucles.
2. Muestra las temperaturas en orden cronológico inverso (la más reciente primero).
3. Cuenta cuántos días superaron los 33 °C y muestra esas temperaturas.
4. Reorganiza los datos en una tabla de 2 semanas x 7 días y calcula la media de cada semana.
5. Obtén, sin repetir y ordenados, los días de la semana (0 a 6) en que alguna de las dos semanas pasó de 33 °C.

### Solución

In [16]:
maximas = np.array([31.2, 29.8, 33.5, 35.1, 34.0, 28.7, 27.9,
                    30.4, 32.6, 36.2, 35.8, 29.1, 28.3, 30.0])

fahrenheit = maximas * 9 / 5 + 32
print("1. En °F:", np.round(fahrenheit, 1))

print("2. Inverso:", maximas[::-1])

calurosos = maximas > 33
print(f"3. {calurosos.sum()} días sobre 33 °C:", maximas[calurosos])

semanas = maximas.reshape(2, 7)
print("4. Media por semana:", np.round(semanas.mean(axis=1), 2))

# np.nonzero devuelve las posiciones donde la máscara vale True
dias_semana_1 = np.nonzero(semanas[0] > 33)[0]
dias_semana_2 = np.nonzero(semanas[1] > 33)[0]
print("5. Días de la semana:", np.union1d(dias_semana_1, dias_semana_2))

1. En °F: [88.2 85.6 92.3 95.2 93.2 83.7 82.2 86.7 90.7 97.2 96.4 84.4 82.9 86. ]
2. Inverso: [30.  28.3 29.1 35.8 36.2 32.6 30.4 27.9 28.7 34.  35.1 33.5 29.8 31.2]
3. 5 días sobre 33 °C: [33.5 35.1 34.  36.2 35.8]
4. Media por semana: [31.46 31.77]
5. Días de la semana: [2 3 4]


## Resumen

- Un `ndarray` guarda valores de un solo `dtype`; su `shape` describe sus dimensiones.
- Las operaciones **vectorizadas** y el **broadcasting** sustituyen a los bucles y son mucho más rápidos.
- El slicing (incluido el paso negativo `[::-1]`) devuelve vistas; `.copy()` crea un arreglo independiente.
- Las **máscaras booleanas** filtran datos; se combinan con `&`, `|` y `~`.
- `np.unique`, `np.union1d` e `np.intersect1d` resuelven operaciones de conjuntos y ya devuelven el resultado ordenado.
- En las agregaciones 2D, `axis=0` da un resultado por columna y `axis=1`, uno por fila.

En el **Módulo 09** se presenta la visualización de datos con matplotlib: histogramas, diagramas de caja y gráficos de barras.